In [ ]:
%run ../_init_notebook.py

# BTC Monthly Seasonality

Análisis exhaustivo del comportamiento de BTC-USD por mes del año.

Bitcoin tiene ciclos marcados (halvings, bull/bear markets) que pueden sesgar los promedios.
El objetivo es entender si la estacionalidad mensual es **robusta entre ciclos** o solo un artefacto de pocos años.

## 1. Carga de datos y cálculo de retornos mensuales

In [ ]:
btc_path = os.path.join(PROJECT_ROOT, 'data', 'yahoo_1d', 'cripto', 'BTC-USD.csv')

df = pd.read_csv(btc_path, header=[0, 1], index_col=0, parse_dates=True)
df.columns = [col[0] for col in df.columns]
df = df.sort_index()

# Retorno mensual: primer close del mes vs último close del mes
monthly = df['Close'].resample('ME').agg(['first', 'last'])
monthly.columns = ['open_month', 'close_month']
monthly['ret_pct'] = (monthly['close_month'] / monthly['open_month'] - 1) * 100
monthly['year'] = monthly.index.year
monthly['month'] = monthly.index.month
monthly['month_name'] = monthly.index.strftime('%b')

# Excluir meses incompletos al inicio/fin
monthly = monthly.dropna()

print(f"Rango: {monthly.index[0].strftime('%Y-%m')} → {monthly.index[-1].strftime('%Y-%m')}")
print(f"Total de meses: {len(monthly)}")
monthly.head()

## 2. Heatmap año × mes — la foto completa

In [ ]:
MONTH_ORDER = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

pivot = monthly.pivot_table(index='year', columns='month', values='ret_pct')
pivot.columns = MONTH_ORDER[:len(pivot.columns)]

# Añadir columna de retorno anual
annual = df['Close'].resample('YE').agg(['first', 'last'])
annual['ret_pct'] = (annual['last'] / annual['first'] - 1) * 100
annual.index = annual.index.year
pivot['YEAR'] = annual['ret_pct']

fig, ax = plt.subplots(figsize=(16, 8))

# Separar columnas de meses y año total
month_cols = MONTH_ORDER
data_months = pivot[month_cols]

vmax = 60
sns.heatmap(
    data_months, annot=True, fmt='.0f', center=0,
    cmap='RdYlGn', linewidths=0.5, linecolor='#cccccc',
    vmin=-vmax, vmax=vmax, ax=ax,
    cbar_kws={'label': '% retorno mensual'}
)

# Overlay columna retorno anual al costado
for i, (yr, row) in enumerate(pivot.iterrows()):
    color = '#1a6620' if row['YEAR'] >= 0 else '#8b0000'
    ax.text(len(month_cols) + 0.5, i + 0.5, f"{row['YEAR']:.0f}%",
            ha='center', va='center', fontsize=9, color=color, fontweight='bold')

ax.text(len(month_cols) + 0.5, -0.6, 'YEAR', ha='center', va='center', fontsize=9, fontweight='bold')
ax.set_title('BTC-USD: Retorno mensual (%) por año', fontsize=14, pad=20)
ax.set_xlabel('')
ax.set_ylabel('Año')
plt.tight_layout()
plt.show()

## 3. Estadísticas por mes — promedio, mediana, win rate, distribución

In [ ]:
stats = monthly.groupby('month')['ret_pct'].agg(
    mean='mean',
    median='median',
    std='std',
    min='min',
    max='max',
    count='count'
).round(2)

stats['win_rate'] = monthly.groupby('month')['ret_pct'].apply(lambda x: (x > 0).mean() * 100).round(1)
stats.index = MONTH_ORDER
stats.index.name = 'Month'

# Ordenar por mean para ver el ranking
print("Estadísticas por mes (ordenado por retorno promedio):")
display(stats.sort_values('mean', ascending=False))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# --- Plot 1: Promedio y mediana por mes ---
x = np.arange(12)
width = 0.35
mean_colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in stats['mean']]
median_colors = ['#27ae60' if v >= 0 else '#c0392b' for v in stats['median']]

bars1 = axes[0].bar(x - width/2, stats['mean'], width, label='Promedio', color=mean_colors, alpha=0.85)
bars2 = axes[0].bar(x + width/2, stats['median'], width, label='Mediana', color=median_colors, alpha=0.5, edgecolor='black', linewidth=0.5)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(MONTH_ORDER)
axes[0].set_ylabel('Retorno (%)')
axes[0].set_title('BTC-USD: Retorno promedio y mediana por mes')
axes[0].legend()

# Anotar valores
for bar in bars1:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, h + (0.5 if h >= 0 else -1.5),
                 f'{h:.1f}%', ha='center', va='bottom', fontsize=7)

# --- Plot 2: Win rate por mes ---
wr_colors = ['#2ecc71' if v >= 50 else '#e74c3c' for v in stats['win_rate']]
axes[1].bar(x, stats['win_rate'], color=wr_colors, alpha=0.85)
axes[1].axhline(50, color='gray', linestyle='--', linewidth=1, label='50%')
axes[1].set_xticks(x)
axes[1].set_xticklabels(MONTH_ORDER)
axes[1].set_ylabel('Win rate (%)')
axes[1].set_title('BTC-USD: % de años donde el mes cerró en positivo (Win Rate)')
axes[1].set_ylim(0, 100)
axes[1].legend()

for i, v in enumerate(stats['win_rate']):
    axes[1].text(i, v + 1.5, f'{v:.0f}%', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

## 4. Box plots — distribución completa de retornos por mes

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

data_by_month = [monthly[monthly['month'] == m]['ret_pct'].values for m in range(1, 13)]

bp = ax.boxplot(data_by_month, patch_artist=True, labels=MONTH_ORDER,
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(linewidth=1.2),
                flierprops=dict(marker='o', markersize=4, alpha=0.5))

for i, (patch, med_val) in enumerate(zip(bp['boxes'], stats['median'])):
    patch.set_facecolor('#2ecc71' if med_val >= 0 else '#e74c3c')
    patch.set_alpha(0.6)

# Scatter de puntos individuales (cada año)
for i, data in enumerate(data_by_month):
    jitter = np.random.normal(0, 0.08, size=len(data))
    ax.scatter(np.full_like(data, i + 1) + jitter, data,
               alpha=0.5, s=25, color='#2c3e50', zorder=3)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_ylabel('Retorno mensual (%)')
ax.set_title('BTC-USD: Distribución de retornos mensuales (cada punto = 1 año)')
plt.tight_layout()
plt.show()

## 5. Ciclos de Bitcoin: bull vs bear años

Clasificamos cada año según el retorno anual para ver si la estacionalidad se mantiene en distintos ciclos.

In [ ]:
# Clasificación de años
annual_ret = monthly.groupby('year')['ret_pct'].sum()  # aproximación: suma de retornos mensuales

# Retorno real anual (year-over-year)
annual_real = df['Close'].resample('YE').last().pct_change() * 100
annual_real.index = annual_real.index.year
annual_real = annual_real.dropna()

bull_years = annual_real[annual_real > 0].index.tolist()
bear_years = annual_real[annual_real <= 0].index.tolist()

print(f"Anos BULL ({len(bull_years)}): {bull_years}")
print(f"Anos BEAR ({len(bear_years)}): {bear_years}")

monthly['cycle'] = monthly['year'].apply(lambda y: 'Bull' if y in bull_years else 'Bear')

# Estadísticas por ciclo
bull_stats = monthly[monthly['cycle'] == 'Bull'].groupby('month')['ret_pct'].mean()
bear_stats = monthly[monthly['cycle'] == 'Bear'].groupby('month')['ret_pct'].mean()
bull_stats.index = MONTH_ORDER
bear_stats.index = MONTH_ORDER

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for ax, data, title, color in [
    (axes[0], bull_stats, f'Anos BULL ({len(bull_years)})', '#2ecc71'),
    (axes[1], bear_stats, f'Anos BEAR ({len(bear_years)})', '#e74c3c'),
]:
    bar_colors = [color if v >= 0 else ('#c0392b' if color == '#2ecc71' else '#7f8c8d') for v in data]
    ax.bar(MONTH_ORDER, data, color=bar_colors, alpha=0.85)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'Retorno promedio por mes — {title}')
    ax.set_ylabel('Retorno promedio (%)')
    for i, v in enumerate(data):
        ax.text(i, v + (0.5 if v >= 0 else -1.5), f'{v:.1f}%', ha='center', fontsize=7)

plt.suptitle('BTC-USD: Estacionalidad mensual — Bull vs Bear', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 6. Foco: Abril y Agosto — ¿cuán consistentes son?


In [ ]:
for month_num, month_name in [(4, 'Abril'), (8, 'Agosto')]:
    data = monthly[monthly['month'] == month_num][['year', 'ret_pct', 'cycle']].sort_values('year')
    
    print(f"\n{'='*50}")
    print(f"  {month_name} — retorno histórico año a año")
    print(f"{'='*50}")
    print(f"  Promedio:  {data['ret_pct'].mean():.2f}%")
    print(f"  Mediana:   {data['ret_pct'].median():.2f}%")
    print(f"  Std:       {data['ret_pct'].std():.2f}%")
    print(f"  Win rate:  {(data['ret_pct'] > 0).mean()*100:.1f}%  ({(data['ret_pct'] > 0).sum()}/{len(data)} años positivos)")
    print(f"  Bull years promedio: {data[data['cycle']=='Bull']['ret_pct'].mean():.2f}%")
    print(f"  Bear years promedio: {data[data['cycle']=='Bear']['ret_pct'].mean():.2f}%")
    print()
    display(data.set_index('year'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, month_num, month_name in [(axes[0], 4, 'Abril'), (axes[1], 8, 'Agosto')]:
    data = monthly[monthly['month'] == month_num].sort_values('year')
    bar_colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in data['ret_pct']]
    ax.bar(data['year'], data['ret_pct'], color=bar_colors, alpha=0.85)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axhline(data['ret_pct'].mean(), color='orange', linestyle='--', linewidth=1.5,
               label=f"Media: {data['ret_pct'].mean():.1f}%")
    ax.set_title(f'BTC-USD — {month_name} por año')
    ax.set_xlabel('Año')
    ax.set_ylabel('Retorno (%)')
    ax.legend()
    for i, row in data.iterrows():
        ax.text(row['year'], row['ret_pct'] + (0.5 if row['ret_pct'] >= 0 else -2),
                f"{row['ret_pct']:.0f}%", ha='center', fontsize=7, rotation=45)

plt.tight_layout()
plt.show()

## 7. Ranking de meses — consistencia entre ciclos

In [ ]:
# Tabla resumen: promedio global, bull, bear + win rate
summary = pd.DataFrame({
    'mean_all': monthly.groupby('month')['ret_pct'].mean(),
    'median_all': monthly.groupby('month')['ret_pct'].median(),
    'win_rate': monthly.groupby('month')['ret_pct'].apply(lambda x: (x > 0).mean() * 100),
    'mean_bull': monthly[monthly['cycle']=='Bull'].groupby('month')['ret_pct'].mean(),
    'mean_bear': monthly[monthly['cycle']=='Bear'].groupby('month')['ret_pct'].mean(),
    'n': monthly.groupby('month')['ret_pct'].count(),
}).round(2)
summary.index = MONTH_ORDER
summary.index.name = 'Month'

# Rank por mean_all
summary['rank'] = summary['mean_all'].rank(ascending=False).astype(int)

print("Ranking de meses por retorno promedio global:")
display(summary.sort_values('mean_all', ascending=False))

# Heatmap de resumen
fig, ax = plt.subplots(figsize=(10, 5))
display_cols = ['mean_all', 'median_all', 'win_rate', 'mean_bull', 'mean_bear']
col_labels = ['Prom. global', 'Mediana', 'Win rate %', 'Prom. bull', 'Prom. bear']
heat_data = summary[display_cols].copy()
heat_data.columns = col_labels

sns.heatmap(heat_data, annot=True, fmt='.1f', center=0,
            cmap='RdYlGn', linewidths=0.5, ax=ax,
            cbar_kws={'label': '%'})
ax.set_title('BTC-USD: Resumen de estacionalidad mensual', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Drawdown intra-mes vs cierre: ¿cómo llega BTC al final del mes?

¿El cierre positivo de Abril viene con mucha volatilidad intra-mes o es un movimiento limpio?

In [ ]:
# Calcular max drawdown intra-mes y máximo upside intra-mes
def intra_month_stats(group):
    close = group['Close']
    open_price = close.iloc[0]
    running_max = close.expanding().max()
    max_upside = ((close.max() / open_price) - 1) * 100
    max_drawdown = ((close.min() / open_price) - 1) * 100
    final_ret = ((close.iloc[-1] / open_price) - 1) * 100
    return pd.Series({'max_upside': max_upside, 'max_drawdown': max_drawdown, 'final_ret': final_ret})

intra = df.groupby([df.index.year, df.index.month]).apply(intra_month_stats)
intra.index.names = ['year', 'month']
intra = intra.reset_index()
intra['month_name'] = intra['month'].apply(lambda m: MONTH_ORDER[m-1])

by_month_intra = intra.groupby('month')[['max_upside', 'max_drawdown', 'final_ret']].mean().round(2)
by_month_intra.index = MONTH_ORDER

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(12)
w = 0.25
ax.bar(x - w, by_month_intra['final_ret'], w, label='Retorno final', color='#3498db', alpha=0.85)
ax.bar(x, by_month_intra['max_upside'], w, label='Max upside intra-mes', color='#2ecc71', alpha=0.85)
ax.bar(x + w, by_month_intra['max_drawdown'], w, label='Max drawdown intra-mes', color='#e74c3c', alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(MONTH_ORDER)
ax.set_ylabel('% desde apertura del mes')
ax.set_title('BTC-USD: Retorno final vs rango intra-mes (promedio histórico)')
ax.legend()
plt.tight_layout()
plt.show()

display(by_month_intra)